In [19]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
from sklearn import preprocessing
from sklearn.metrics import r2_score, mean_absolute_error
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import random
from pathlib import Path
import os
from datetime import datetime
import pickle

## Parameters

In [20]:
data_file = os.path.join(os.getcwd(), "PACE_formatted_data.csv")
model_save_dir = os.path.join(os.getcwd(), "FF_models")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_out = 3    # 3 output parameters: MEF, MDF, interecept. 
             # inclusion of intercept is debatable (we'd love to have a model where it is zero),
             # and so far it has not much impacted r^2.
             # once we do test train split i only want to keep it if it notably improves model accuracy

In [21]:
device

device(type='cpu')

## data exploration

In [22]:
data_df = pd.read_csv(data_file)

In [23]:
data_df.head()

,Interval Start,Internal CO2 (tonnes),Exported CO2 (tonnes),Imported CO2 (tonnes),Load,VRE,Total_CO2_Emissions,delta_Load,delta_VRE,delta_Total_CO2_Emissions
0,2023-01-01 08:00:00+00:00,4666.804580,1309.263000,1432.543998,5593.0,134.0,4790.085578,-38.0,-185.0,150.530114
1,2023-01-01 09:00:00+00:00,4611.856384,1386.782865,1449.931106,5429.0,138.0,4675.004625,-164.0,4.0,-115.080953
2,2023-01-01 10:00:00+00:00,4586.727360,1361.146724,1436.326960,5391.0,148.0,4661.907595,-38.0,10.0,-13.097030
3,2023-01-01 11:00:00+00:00,4607.864771,1411.044621,1384.964866,5408.0,215.0,4581.785016,17.0,67.0,-80.122579
4,2023-01-01 12:00:00+00:00,4499.365445,1300.808943,1308.941856,5453.0,339.0,4507.498358,45.0,124.0,-74.286658


## Data Loading and pre-processing

In [24]:
PACE_Data = pd.read_csv(data_file, index_col=0)
PACE_Data.index = pd.to_datetime(PACE_Data.index)

In [25]:
PACE_Data.index[0]

Timestamp('2023-01-01 08:00:00+0000', tz='UTC')

In [26]:
# number of hours since earliest data point
(PACE_Data.index[1] - PACE_Data.index[0]).total_seconds() // 3600

1.0

In [27]:
# read in the processed data. there should be zero nans or missing data.
PACE_Data = pd.read_csv(data_file, index_col=0)
PACE_Data.index = pd.to_datetime(PACE_Data.index)

### Feature engineering...

## Temporal features
PACE_Data.loc[:,"Day_of_Year"] = [instant.timetuple().tm_yday for instant in PACE_Data.index]
PACE_Data.loc[:,"Hour"] = PACE_Data.index.hour

# number of hours since earliest data point
def hours_since_2022(instant):
    return (instant - pd.Timestamp(2023, 1, 1, 0, 0, tz='UTC')).total_seconds() // 3600
PACE_Data.loc[:,"Hours_since_2022"] = [hours_since_2022(instant) for instant in PACE_Data.index]
# PACE_Data.loc[:,"Is_Weekend"] = [instant.weekday() > 4 for instant in PACE_Data.index]
# for day_of_week in range(7):
#     PACE_Data.loc[:,f"Day_of_Week={day_of_week}"] = [instant.weekday() ==day_of_week for instant in PACE_Data.index]

## Recent history features
# prev_time_steps = [1,2]#,24,3]
# for i in prev_time_steps:
#     PACE_Data.loc[:,f"Load_t-{i}"] = ([np.nan] * i) + list(PACE_Data.iloc[:-i].loc[:,"Load"])
#     PACE_Data.loc[:,f"VRE_t-{i}"] = ([np.nan] * i) + list(PACE_Data.iloc[:-i].loc[:,"VRE"])
#     # PACE_Data.loc[:,f"d_emissions_t-{i}"] = ([np.nan] * i) + list(PACE_Data.iloc[:-i].loc[:,"delta_Total_CO2_Emissions"])
#     PACE_Data.loc[:,f"Net_load_t-{i}"] = ([np.nan] * i) + list(PACE_Data.iloc[:-i].loc[:,"Net Load"])
# # Drop rows that don't have all prev-time-step features
# PACE_Data = PACE_Data.iloc[max(prev_time_steps):]


## load and VRE derivatives
# PACE_Data.loc[:,"Load_d1"] = [np.nan] + list(PACE_Data.iloc[1:].loc[:,"Load"].values - PACE_Data.iloc[:-1].loc[:,"Load"].values)
# PACE_Data.loc[:,"Load_d2"] = [np.nan, np.nan] + list(PACE_Data.iloc[2:].loc[:,"Load_d1"].values - PACE_Data.iloc[1:-1].loc[:,"Load_d1"].values)
# PACE_Data.loc[:,"VRE_d1"] = [np.nan] + list(PACE_Data.iloc[1:].loc[:,"VRE"].values - PACE_Data.iloc[:-1].loc[:,"VRE"].values)
# PACE_Data.loc[:,"VRE_d2"] = [np.nan, np.nan] + list(PACE_Data.iloc[2:].loc[:,"VRE_d1"].values - PACE_Data.iloc[1:-1].loc[:,"VRE_d1"].values)
#PACE_Data.loc[:,"Net_Load_d1"] = [np.nan] + list(PACE_Data.iloc[1:].loc[:,"Net Load"].values - PACE_Data.iloc[:-1].loc[:,"Net Load"].values)
# PACE_Data.loc[:,"Net_Load_d2"] = [np.nan, np.nan] + list(PACE_Data.iloc[2:].loc[:,"Net_Load_d1"].values - PACE_Data.iloc[1:-1].loc[:,"Net_Load_d1"].values)
# # Drop rows that don't have all features
#PACE_Data = PACE_Data.iloc[1:]


num_samples = len(PACE_Data)

# create masks for train, validation and test data
percent_train, percent_val, percent_test = .6, .2, .2
num_train, num_val = int(percent_train * num_samples), int(percent_val * num_samples)
num_test = num_samples - num_train - num_val
set_assignments = np.array([1 for i in range(num_train)] + [2 for i in range(num_val)] + [3 for i in range(num_test)])
np.random.seed(1)
np.random.shuffle(set_assignments)
train_mask = set_assignments == 1
val_mask = set_assignments == 2
test_mask = set_assignments == 3
PACE_train = PACE_Data.loc[train_mask]
PACE_val = PACE_Data.loc[val_mask]
PACE_test = PACE_Data.loc[test_mask]

## old features we've tried
# feature_cols = ['Load', 'VRE', 'Hour', 'Month', 'Day', 'Pas_Temp', 'SJ_Temp', 'NH_Temp', 'SB_Temp', 'Sac_Temp', 'Fres_Temp', 'LB_Temp']
# feature_cols = ['Load', 'VRE', 'Hour', 'Day_of_Year', 'Is_Weekend', 'Pas_Temp', 'SJ_Temp', 'NH_Temp', 'SB_Temp', 'Sac_Temp', 'Fres_Temp', 'LB_Temp']
# feature_cols = ['Load', 'VRE', 'Hour', 'Day_of_Year', 'Pas_Temp', 'SJ_Temp', 'NH_Temp', 'SB_Temp', 'Sac_Temp', 'Fres_Temp', 'LB_Temp']
# feature_cols.extend([f"Day_of_Week={day_of_week}" for day_of_week in range(7)])

feature_cols = ['Load', 'VRE', 'Hour', 'Day_of_Year', "Hours_since_2022"]
# feature_cols.extend([f"Load_t-{i}" for i in prev_time_steps] + [f"VRE_t-{i}" for i in prev_time_steps])
# feature_cols.extend(["Load_d1", "Load_d2", "VRE_d1", "VRE_d2"])
# feature_cols.extend(["delta_Load", "delta_VRE"])
# feature_cols.extend([f"d_emissions_t-{i}" for i in prev_time_steps])
# feature_cols.extend([f"Net_load_t-{i}" for i in prev_time_steps])
# feature_cols.extend(["Net_Load_d1"])

# specify x and y data #TODO seems like there may be some unnecessary operations here... 
train_x = torch.tensor(PACE_train[feature_cols].values.astype(np.float32))
val_x = torch.tensor(PACE_val[feature_cols].values.astype(np.float32))
test_x = torch.tensor(PACE_test[feature_cols].values.astype(np.float32))
# train_y = torch.tensor(np.array(PACE_train['delta_Total_CO2_Emissions'].values).astype(np.float32))
# val_y = torch.tensor(np.array(PACE_val['delta_Total_CO2_Emissions'].values).astype(np.float32))
# test_y = torch.tensor(np.array(PACE_test['delta_Total_CO2_Emissions'].values).astype(np.float32))


# standardize data based on mean and variance of train data
scaler = preprocessing.StandardScaler()
scaler.fit(train_x)
train_x = torch.tensor(scaler.transform(train_x)).to(device)
val_x = torch.tensor(scaler.transform(val_x)).to(device)
test_x = torch.tensor(scaler.transform(test_x)).to(device)

# save scaler for later re-use
with open(os.path.join(model_save_dir, "scaler.hours_since_2022.pkl"), 'wb') as f:
    pickle.dump(scaler, f)

In [28]:
PACE_Data.iloc[:5]

,Internal CO2 (tonnes),Exported CO2 (tonnes),Imported CO2 (tonnes),Load,VRE,Total_CO2_Emissions,delta_Load,delta_VRE,delta_Total_CO2_Emissions,Day_of_Year,Hour,Hours_since_2022
Interval Start,,,,,,,,,,,,
2023-01-01 08:00:00+00:00,4666.804580,1309.263000,1432.543998,5593.0,134.0,4790.085578,-38.0,-185.0,150.530114,1,8,8.0
2023-01-01 09:00:00+00:00,4611.856384,1386.782865,1449.931106,5429.0,138.0,4675.004625,-164.0,4.0,-115.080953,1,9,9.0
2023-01-01 10:00:00+00:00,4586.727360,1361.146724,1436.326960,5391.0,148.0,4661.907595,-38.0,10.0,-13.097030,1,10,10.0
2023-01-01 11:00:00+00:00,4607.864771,1411.044621,1384.964866,5408.0,215.0,4581.785016,17.0,67.0,-80.122579,1,11,11.0
2023-01-01 12:00:00+00:00,4499.365445,1300.808943,1308.941856,5453.0,339.0,4507.498358,45.0,124.0,-74.286658,1,12,12.0


In [29]:
train_x.shape

torch.Size([15777, 5])

In [30]:
feature_cols

['Load', 'VRE', 'Hour', 'Day_of_Year', 'Hours_since_2022']

In [31]:
train_x[:,-1]

tensor([-1.7203, -1.7200, -1.7199,  ...,  1.7349,  1.7351,  1.7352],
       dtype=torch.float64)

## define model loss function

In [32]:
def get_model(n_input, hidden_dims, n_out, dropout_p):
    
    layers = [nn.Linear(n_input, hidden_dims[0]),
              nn.BatchNorm1d(hidden_dims[0]),
              nn.ReLU(),
              nn.Dropout(dropout_p)
             ]
    for layer in range(len(hidden_dims)-1):
        cur_hidden, next_hidden = hidden_dims[layer], hidden_dims[layer+1]
        layers.extend([nn.Linear(cur_hidden, next_hidden),
                       nn.BatchNorm1d(next_hidden),
                       nn.ReLU(),
                       nn.Dropout(dropout_p)
                      ])
    layers.append(nn.Linear(hidden_dims[-1], n_out))
    
    model = nn.Sequential(*layers)
    return model

def mse_loss_regularized_preds_l2(pred_coeff, PACE_Data, MEF_reg_weight, MDF_reg_weight, bias_term):
    delta_load_tensor = torch.tensor(np.array(PACE_Data['delta_Load'].values).astype(np.float32))
    delta_vre_tensor = torch.tensor(np.array(PACE_Data['delta_VRE'].values).astype(np.float32))
    MEF_preds = pred_coeff[:,0]
    MDF_preds = pred_coeff[:,1]
    pred_y_demand = torch.mul(delta_load_tensor, MEF_preds)
    pred_y_vre = torch.mul(delta_vre_tensor, MDF_preds)
    pred_y = pred_y_vre + pred_y_demand
    if bias_term:
        bias_preds = pred_coeff[:,2]
        pred_y += bias_preds
    act_y = torch.tensor(np.array(PACE_Data['delta_Total_CO2_Emissions'].values).astype(np.float32))

    # Compute MEF regularization term (sum(MEF^2 + intercept if MEF < 0 for MEF in examples))
    invalid_MEFs = nn.functional.relu(-MEF_preds)  # keep negative MEFs and zero others
    count_invalid_MEFs = torch.count_nonzero(invalid_MEFs)
    MEF_reg_intercept = 4420  # based on an average value of -65 seen amongst invalids when trained without regularization
    MEF_reg = torch.dot(invalid_MEFs, invalid_MEFs) + (count_invalid_MEFs * MEF_reg_intercept)

    # Compute MDF regularization term (sum(MDF^2 + intercept if MDF > 0 for MDF in examples))
    invalid_MDFs = nn.functional.relu(MDF_preds)  # keep negative MEFs and zero others
    count_invalid_MDFs = torch.count_nonzero(invalid_MDFs)
    MDF_reg_intercept = 538  # based on an average value of +23 seen amongst invalids when trained without regularization
    MDF_reg = torch.dot(invalid_MDFs, invalid_MDFs) + (count_invalid_MDFs * MDF_reg_intercept)

    loss = nn.MSELoss()(pred_y, act_y) + (MEF_reg_weight * MEF_reg) + (MDF_reg_weight * MDF_reg)
    return loss

def mse_loss_regularized_preds_l1(pred_coeff, PACE_Data, MEF_reg_weight, MDF_reg_weight, bias_term):
    delta_load_tensor = torch.tensor(np.array(PACE_Data['delta_Load'].values).astype(np.float32))
    delta_vre_tensor = torch.tensor(np.array(PACE_Data['delta_VRE'].values).astype(np.float32))
    MEF_preds = pred_coeff[:,0]
    MDF_preds = pred_coeff[:,1]
    pred_y_demand = torch.mul(delta_load_tensor, MEF_preds)
    pred_y_vre = torch.mul(delta_vre_tensor, MDF_preds)
    pred_y = pred_y_vre + pred_y_demand
    if bias_term:
        bias_preds = pred_coeff[:,2]
        pred_y += bias_preds
    act_y = torch.tensor(np.array(PACE_Data['delta_Total_CO2_Emissions'].values).astype(np.float32))

    # Compute MEF regularization term (sum(MEF + intercept if MEF < 0 for MEF in examples))
    invalid_MEFs = nn.functional.relu(-MEF_preds)  # keep negative MEFs and zero others
    count_invalid_MEFs = torch.count_nonzero(invalid_MEFs)
    MEF_reg_intercept = 66.5  # The average value seen amongst invalids when trained without regularization
    MEF_reg = torch.sum(invalid_MEFs) + (count_invalid_MEFs * MEF_reg_intercept)

    # Compute MDF regularization term (sum(MDF + intercept if MDF > 0 for MDF in examples))
    invalid_MDFs = nn.functional.relu(MDF_preds)  # keep negative MEFs and zero others
    count_invalid_MDFs = torch.count_nonzero(invalid_MDFs)
    MDF_reg_intercept = 23.2  # The average value seen amongst invalids when trained without regularization
    MDF_reg = torch.sum(invalid_MDFs) + (count_invalid_MDFs * MDF_reg_intercept)

    loss = nn.MSELoss()(pred_y, act_y) + (MEF_reg_weight * MEF_reg) + (MDF_reg_weight * MDF_reg)
    return loss

## Model Training

Helpers for printing results

In [33]:
pd.options.mode.chained_assignment = None  # default='warn'

In [34]:
def plot_losses(train_losses, val_losses, plt_save_dir=None):
    #plot loss vs epochs
    fig, axs = plt.subplots(1,2)
    axs[0].plot(train_losses[1000:])
    axs[0].set_title("Train Set")
    axs[0].set_ylabel('loss')
    axs[0].set_xlabel('epoch')

    axs[1].plot(val_losses[1000:])
    axs[1].set_title("Val Set")
    axs[1].set_ylabel('loss')
    axs[1].set_xlabel('epoch')
    
    plt.tight_layout()
    plt.show()
    
    if plt_save_dir:
        fig.savefig(os.path.join(plt_save_dir,"train_val_losses.png"))
        
def get_r_squared(pred_coeff, PACE_Data, bias_term):
    coeff_df=pd.DataFrame(data=pred_coeff.detach().numpy(), columns=['MEF', 'MDF', 'Intercept'])
    pred_delta_total_co2_emissions = coeff_df['MEF'].values * PACE_Data['delta_Load'] \
                                   + coeff_df['MDF'].values * PACE_Data['delta_VRE']
    if bias_term:
        pred_delta_total_co2_emissions += coeff_df['Intercept'].values
    r2 = r2_score(PACE_Data['delta_Total_CO2_Emissions'], pred_delta_total_co2_emissions)
    return r2

def get_mean_abs_err(pred_coeff, PACE_Data, bias_term):
    coeff_df=pd.DataFrame(data=pred_coeff.detach().numpy(), columns=['MEF', 'MDF', 'Intercept'])
    pred_delta_total_co2_emissions = coeff_df['MEF'].values * PACE_Data['delta_Load'] \
                                   + coeff_df['MDF'].values * PACE_Data['delta_VRE']
    if bias_term:
        pred_delta_total_co2_emissions += coeff_df['Intercept'].values
    mean_abs_err = mean_absolute_error(PACE_Data['delta_Total_CO2_Emissions'], pred_delta_total_co2_emissions)
    return mean_abs_err

def get_count_invalid_preds(pred_coeff):
    # preds=pred_coeff.detach().numpy()
    count_neg_MEFs = torch.sum(pred_coeff[:,0] <= 0).item() #sum(preds[:,0] <= 0) # MEF must be greater than 0
    count_pos_MDFs = torch.sum(pred_coeff[:,1] > 0).item() #sum(preds[:,1] > 0)  # MDF must be less than or equal to 0
    return count_neg_MEFs, count_pos_MDFs

Helper for training a model with a given set of hyperparameters and saving the best model and results

In [35]:
def train_model_with_params(train_x, val_x, PACE_train, PACE_val, n_out, hidden_dims, learning_rate, weight_decay, dropout_p, \
                            loss_function, MEF_reg_weight, MDF_reg_weight, bias_term, model_dir_prefix=None, epochs=10000, min_save_r2=.87):
    
    n_input = train_x.shape[1]
    model = get_model(n_input, hidden_dims, n_out, dropout_p)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    model_dir = model_save_dir
    if model_dir_prefix:
        model_dir = os.path.join(model_dir, model_dir_prefix)
    model_dir = os.path.join(model_dir, str(datetime.now().strftime('%Y_%m_%d-%I_%M_%S_%p')))
    Path(model_dir).mkdir(parents=True, exist_ok=True)
    
    # log experiment settings
    settings_str = "Model Settings:"
    settings_str += f"\n\t{n_input=}\n\t{hidden_dims=}\n\t{n_out=}\n\t{dropout_p=}\n\t{bias_term=}"
    settings_str += "\nOptimizer Settings:"
    settings_str += f"\n\t{learning_rate=}\n\t{weight_decay=}"
    settings_str += "\nLoss Function Settings:"
    settings_str += f"\n\t{loss_function=}\n\t{MEF_reg_weight=}\n\t{MDF_reg_weight=}"
    settings_str += "\nTrain Process Settings:"
    settings_str += f"\n\t{epochs=}\n\t{min_save_r2=}"
    settings_str += f"\nFeatures: {', '.join(feature_cols)}"
    print(settings_str)
    with open(os.path.join(model_dir, "experiment_settings.txt"), 'w+') as f:
        f.write(settings_str)
        
    model.to(device)

    best_r2 = -np.inf 
    best_epoch = None
    save_model_path = None
    last_save_epoch = None
    last_save_r2 = -np.inf
    last_save_mae = np.inf
    
    train_losses = []
    val_losses = []
    for epoch in tqdm(range(epochs)):
        # tell model we are training (for bathnorm layer, dropout...)
        model.train()
        train_pred_coeff=model(train_x.float()).cpu()
        train_loss=loss_function(train_pred_coeff, PACE_train, MEF_reg_weight, MDF_reg_weight, bias_term)
        train_losses.append(train_loss.item())
        
        # tell model we are evaluating
        model.eval()
        val_pred_coeff=model(val_x.float()).cpu()
        val_loss=loss_function(val_pred_coeff, PACE_val, MEF_reg_weight, MDF_reg_weight, bias_term)
        val_losses.append(val_loss.item())
        val_r2 = get_r_squared(val_pred_coeff, PACE_val, bias_term)

        # always keep best r2 updated
        if val_r2 > best_r2:
            best_r2 = val_r2
            best_epoch = epoch
        # check if we should save... we need good enough r2 and no invalids
        if val_r2 > max(last_save_r2, min_save_r2):
            if sum(get_count_invalid_preds(val_pred_coeff))==0:
                # also check training invalids... Let's recompute with eval mode
                model.eval()
                eval_mode_train_preds=model(train_x.float()).cpu()
                if sum(get_count_invalid_preds(eval_mode_train_preds))==0:
                    if save_model_path:
                        Path(save_model_path).unlink() # delete prev-best model
                    model_save_name = f"epoch={epoch},r2={val_r2:.4f},Invalids=0.pth"
                    save_model_path = os.path.join(model_dir, model_save_name)
                    torch.save(model.state_dict(), save_model_path)
                    last_save_epoch = epoch
                    last_save_r2 = val_r2
                    last_save_mae = get_mean_abs_err(val_pred_coeff, PACE_val, bias_term)

        
        if epoch % 1000 == 0:
            invalid_train_MEFs, invalid_train_MDFs = get_count_invalid_preds(train_pred_coeff)
            invalid_val_MEFs, invalid_val_MDFs = get_count_invalid_preds(val_pred_coeff)
            train_r2 = get_r_squared(train_pred_coeff, PACE_train, bias_term)
            # val_r2 = get_r_squared(val_pred_coeff, PACE_val, bias_term)
            print(f"[Epoch {epoch}]")
            print(f"\tTrain Set: Loss={train_loss.item():.3e}, R Squared={train_r2:.4f}, Invalid MEFs={invalid_train_MEFs}, Invalid MDFs={invalid_train_MDFs}")
            print(f"\tVal Set: Loss={val_loss.item():.3e}, R Squared={val_r2:.4f}, Invalid MEFs={invalid_val_MEFs}, Invalid MDFs={invalid_val_MDFs}")
        model.zero_grad()
        train_loss.backward()
        optimizer.step()
        # stop if we aren't improving after 10k epochs
        if best_epoch and epoch > 10000 + best_epoch:
            print("Early stopping as we haven't made an improvement on validation set in 10,000 epochs.")
            break
        
    plot_losses(train_losses, val_losses, model_dir)

    results_str = f"Best R Squared seen on epoch {best_epoch}: {best_r2:.4f}"
    results_str += f"\nBest-R2-model with R2 above {min_save_r2=} and 0 invalid coefficients predicted on train/test sets:"
    if not save_model_path:
        results_str += f"\n\tNo such model was encountered"
    else:
        results_str += f"\n\tValidation R2: {last_save_r2:.4f}"
        results_str += f"\n\tValidation MAE: {last_save_mae:.2f}"
        results_str += f"\n\tEpoch seen: {last_save_epoch}"
        results_str += f"\n\tModel file: {save_model_path.split('/')[-1]}"

    print(results_str)
    with open(os.path.join(model_dir, "results.txt"), 'w+') as f:
        f.write(results_str)
        
    return save_model_path

### Train with varying hyperparams

In [ ]:
hidden_dim_settings = [[512,256]]
lrs = [.005, .01]
weight_decays = [.001, .003]
dropout_probs = [0.5]
bias_term = True
regularization_weights = [[0, 0]]  # set to [1e6, 1e6] after calibrating intercepts
loss_function = mse_loss_regularized_preds_l1
save_prefix = "hours_since_2018_feature"

for hidden_dims in hidden_dim_settings:
    for lr in lrs:
        for weight_decay in weight_decays:
            for dropout_p in dropout_probs:
                for reg_weights in regularization_weights:
                    train_model_with_params(train_x, val_x, PACE_train, PACE_val, n_out, hidden_dims, lr, weight_decay, dropout_p,
                                            loss_function, *reg_weights, bias_term, save_prefix, epochs=60000, min_save_r2=.88)

Model Settings:
	n_input=5
	hidden_dims=[512, 256]
	n_out=3
	dropout_p=0.5
	bias_term=True
Optimizer Settings:
	learning_rate=0.005
	weight_decay=0.001
Loss Function Settings:
	loss_function=<function mse_loss_regularized_preds_l1 at 0x00000194D56B5760>
	MEF_reg_weight=0
	MDF_reg_weight=0
Train Process Settings:
	epochs=60000
	min_save_r2=0.88
Features: Load, VRE, Hour, Day_of_Year, Hours_since_2022


  0%|          | 0/60000 [00:00<?, ?it/s]

[Epoch 0]
	Train Set: Loss=2.306e+05, R Squared=0.4136, Invalid MEFs=6068, Invalid MDFs=4368
	Val Set: Loss=1.028e+07, R Squared=-13.0372, Invalid MEFs=3328, Invalid MDFs=223


In [ ]:
# Run this after the unregularized training run to measure the correct intercept values for PACE data.
# Then update MEF_reg_intercept and MDF_reg_intercept in mse_loss_regularized_preds_l1,
# set regularization_weights = [[1e6, 1e6]] above, and re-run training.
model.eval()
all_x = torch.cat([train_x, val_x])
with torch.no_grad():
    all_preds = model(all_x.float()).cpu()

invalid_mef_vals = all_preds[:, 0][all_preds[:, 0] < 0]
invalid_mdf_vals = all_preds[:, 1][all_preds[:, 1] > 0]

print(f"Invalid MEFs: count={len(invalid_mef_vals)}, avg magnitude={invalid_mef_vals.abs().mean().item():.2f}")
print(f"Invalid MDFs: count={len(invalid_mdf_vals)}, avg magnitude={invalid_mdf_vals.abs().mean().item():.2f}")
print()
print(f"Set MEF_reg_intercept = {invalid_mef_vals.abs().mean().item():.1f}")
print(f"Set MDF_reg_intercept = {invalid_mdf_vals.abs().mean().item():.1f}")

## Inference on all data

In [19]:
## below we can load specific models instead of best one found in most recent experiment
# best_model_dir = os.path.join(model_save_dir, "base4_features\\2023_01_10-05_36_38_PM")
# best_model_file = os.path.join(best_model_dir, "epoch=37502,r2=0.8856,Invalids=0.pth")

# best_model_dir = os.path.join(model_save_dir, "base4_features\\2023_01_27-04_25_05_PM")
# best_model_file = os.path.join(best_model_dir, "epoch=30059,r2=0.8836,Invalids=0.pth")

# best_model_dir = os.path.join(model_save_dir, "hours_since_2018_feature\\2023_04_01-04_51_40_PM")
# best_model_file = os.path.join(best_model_dir, "epoch=29757,r2=0.8918,Invalids=0.pth")

best_model_dir = os.path.join(model_save_dir, "hours_since_2018_feature\\2023_04_01-05_23_36_PM")
best_model_file = os.path.join(best_model_dir, "epoch=35779,r2=0.8933,Invalids=0.pth")

In [20]:
best_model_file

'c:\\Users\\jacks\\Documents\\ASPIRE\\MEF-Regression\\FF_models\\hours_since_2018_feature\\2023_04_01-05_23_36_PM\\epoch=35779,r2=0.8933,Invalids=0.pth'

In [23]:
hidden_dims = [512,256]
bias_term = True
dropout_p = 0.5
n_input = train_x.shape[1]

model = get_model(n_input, hidden_dims, n_out, dropout_p)
model.to(device)
model.load_state_dict(torch.load(best_model_file, map_location=torch.device('cpu')))
model.eval()

train_pred_coeff = model(train_x.float()).cpu()
val_pred_coeff = model(val_x.float()).cpu()
test_pred_coeff = model(test_x.float()).cpu()

results_str = "R Squared:"
results_str += f"\n\tTrain: {get_r_squared(train_pred_coeff, PACE_train, bias_term):.4f}"
results_str += f"\n\tVal: {get_r_squared(val_pred_coeff, PACE_val, bias_term):.4f}"
results_str += f"\n\tTest: {get_r_squared(test_pred_coeff, PACE_test, bias_term):.4f}"
results_str += "\nMean Absolute Error:"
results_str += f"\n\tTrain: {get_mean_abs_err(train_pred_coeff, PACE_train, bias_term):.2f}"
results_str += f"\n\tVal: {get_mean_abs_err(val_pred_coeff, PACE_val, bias_term):.2f}"
results_str += f"\n\tTest: {get_mean_abs_err(test_pred_coeff, PACE_test, bias_term):.2f}"
results_str += "\nCount Invalid Values Predicted:"
invalid_train_MEFs, invalid_train_MDFs = get_count_invalid_preds(train_pred_coeff)
invalid_val_MEFs, invalid_val_MDFs = get_count_invalid_preds(val_pred_coeff)
invalid_test_MEFs, invalid_test_MDFs = get_count_invalid_preds(test_pred_coeff)
results_str += f"\n\tTrain: Invalid MEFs={invalid_train_MEFs}, Invalid MDFs={invalid_train_MDFs}"
results_str += f"\n\tVal: Invalid MEFs={invalid_val_MEFs}, Invalid MDFs={invalid_val_MDFs}"
results_str += f"\n\tTest: Invalid MEFs={invalid_test_MEFs}, Invalid MDFs={invalid_test_MDFs}"

with open(os.path.join(best_model_dir, "eval_results.txt"), 'w+') as f:
    f.write(results_str)
print(results_str)

R Squared:
	Train: 0.9164
	Val: 0.8933
	Test: 0.8929
Mean Absolute Error:
	Train: 116886.63
	Val: 130745.96
	Test: 126760.97
Count Invalid Values Predicted:
	Train: Invalid MEFs=0, Invalid MDFs=0
	Val: Invalid MEFs=0, Invalid MDFs=0
	Test: Invalid MEFs=0, Invalid MDFs=0


In [25]:
hidden_dims = [512,256]
bias_term = True
dropout_p = 0.5
n_input = train_x.shape[1]

model = get_model(n_input, hidden_dims, n_out, dropout_p)
model.to(device)
model.load_state_dict(torch.load(best_model_file, map_location=torch.device('cpu')))
model.eval()

train_pred_coeff = model(train_x.float()).cpu()
val_pred_coeff = model(val_x.float()).cpu()
test_pred_coeff = model(test_x.float()).cpu()

results_str = "R Squared:"
results_str += f"\n\tTrain: {get_r_squared(train_pred_coeff, PACE_train, bias_term):.4f}"
results_str += f"\n\tVal: {get_r_squared(val_pred_coeff, PACE_val, bias_term):.4f}"
results_str += f"\n\tTest: {get_r_squared(test_pred_coeff, PACE_test, bias_term):.4f}"
results_str += "\nMean Absolute Error:"
results_str += f"\n\tTrain: {get_mean_abs_err(train_pred_coeff, PACE_train, bias_term):.2f}"
results_str += f"\n\tVal: {get_mean_abs_err(val_pred_coeff, PACE_val, bias_term):.2f}"
results_str += f"\n\tTest: {get_mean_abs_err(test_pred_coeff, PACE_test, bias_term):.2f}"
results_str += "\nCount Invalid Values Predicted:"
invalid_train_MEFs, invalid_train_MDFs = get_count_invalid_preds(train_pred_coeff)
invalid_val_MEFs, invalid_val_MDFs = get_count_invalid_preds(val_pred_coeff)
invalid_test_MEFs, invalid_test_MDFs = get_count_invalid_preds(test_pred_coeff)
results_str += f"\n\tTrain: Invalid MEFs={invalid_train_MEFs}, Invalid MDFs={invalid_train_MDFs}"
results_str += f"\n\tVal: Invalid MEFs={invalid_val_MEFs}, Invalid MDFs={invalid_val_MDFs}"
results_str += f"\n\tTest: Invalid MEFs={invalid_test_MEFs}, Invalid MDFs={invalid_test_MDFs}"

with open(os.path.join(best_model_dir, "eval_results.txt"), 'w+') as f:
    f.write(results_str)
print(results_str)

R Squared:
	Train: 0.9164
	Val: 0.8933
	Test: 0.8929
Mean Absolute Error:
	Train: 116886.63
	Val: 130745.96
	Test: 126760.97
Count Invalid Values Predicted:
	Train: Invalid MEFs=0, Invalid MDFs=0
	Val: Invalid MEFs=0, Invalid MDFs=0
	Test: Invalid MEFs=0, Invalid MDFs=0


### Put the MEFs and MDFs from all sets back together and in order into the original DF for viewing

In [26]:
all_preds_w_timestamps = list(zip(PACE_val.index, val_pred_coeff.detach().numpy())) \
                        + list(zip(PACE_train.index, train_pred_coeff.detach().numpy())) \
                        + list(zip(PACE_test.index, test_pred_coeff.detach().numpy()))
all_preds_w_timestamps.sort(key=lambda pair: pair[0])
all_preds_ordered = np.array([pair[1] for pair in all_preds_w_timestamps])

In [27]:
all_MEFs_ordered = all_preds_ordered[:,0]
all_MDFs_ordered = all_preds_ordered[:,1]
all_intercepts_ordered = all_preds_ordered[:,2]

In [28]:
PACE_Data.loc[:,"MEF"] = all_MEFs_ordered
PACE_Data.loc[:,"MDF"] = all_MDFs_ordered
if bias_term:
    PACE_Data.loc[:,"Intercept"] = all_intercepts_ordered

#calculate some error stuff. rn i am thinking R2 is the best measure of error
d_emissions = PACE_Data.loc[:,'MEF'] * PACE_Data.loc[:,'delta_Load'] \
            + PACE_Data.loc[:,'MDF'] * PACE_Data.loc[:,'delta_VRE']
if bias_term:
    d_emissions += PACE_Data.loc[:,"Intercept"]
PACE_Data.loc[:,'Predicted_delta_Total_CO2_Emissions'] = d_emissions
PACE_Data.loc[:,'Error']=PACE_Data.loc[:,'Predicted_delta_Total_CO2_Emissions']-PACE_Data.loc[:,'delta_Total_CO2_Emissions']
PACE_Data.loc[:,'%_Error']=np.abs(PACE_Data.loc[:,'Error'])/np.abs(PACE_Data.loc[:,'delta_Total_CO2_Emissions'])
print("Whole Data Set:")
print(f"\tMean Emissions Change = {np.mean(np.abs(PACE_Data['delta_Total_CO2_Emissions'])):.2f}")
print(f"\tR Squared = {r2_score(PACE_Data['delta_Total_CO2_Emissions'], PACE_Data['Predicted_delta_Total_CO2_Emissions']):.4f}")
print(f"\tMean Absolute Error = {mean_absolute_error(PACE_Data['delta_Total_CO2_Emissions'], PACE_Data['Predicted_delta_Total_CO2_Emissions']):.2f}")

Whole Data Set:
	Mean Emissions Change = 413162.83
	R Squared = 0.9071
	Mean Absolute Error = 121633.36


Add train/val/test set assignments to table before saving

In [29]:
set_assignments

array([3, 1, 1, ..., 1, 1, 1], shape=(26305,))

In [30]:
set_num_to_str = {1: 'train',
                  2: 'validation',
                  3: 'test'
                 }

In [31]:
PACE_Data.loc[:,'set_assignment'] = [set_num_to_str[num] for num in set_assignments]

In [32]:
all(PACE_test.index == PACE_Data.loc[PACE_Data.loc[:,'set_assignment'] == 'test'].index)

True

In [33]:
PACE_Data.head()

,Load,Net Load,Total_CO2_Emissions,Total_SO2_Emissions,Total_NOX_Emissions,VRE,delta_Load,delta_Net_Load,delta_Total_CO2_Emissions,delta_Total_SO2_Emissions,...,Day_of_Year,Hour,Hours_since_2018,MEF,MDF,Intercept,Predicted_delta_Total_CO2_Emissions,Error,%_Error,set_assignment
2019-01-01 00:00:00,22822.964472,20502.358502,5.103942e+06,425.327933,1632.821698,2320.593616,-1285.054865,-1255.110267,-337029.794143,-24.142180,...,1,0,0.0,353.528137,-371.433838,-6224.516113,-449400.771044,-112370.976901,0.333416,test
2019-01-01 01:00:00,21879.620618,19606.836908,4.867578e+06,404.315852,1557.650531,2272.780097,-944.689268,-896.922625,-243021.833700,-21.594332,...,1,1,1.0,343.447418,-378.015411,-3581.675781,-309979.871034,-66958.037335,0.275523,train
2019-01-01 02:00:00,21257.454020,19056.267637,4.723101e+06,383.695714,1496.197481,2201.182455,-614.641020,-545.206677,-144846.797503,-20.952957,...,1,2,2.0,335.775330,-347.679291,-1740.397827,-183981.341478,-39134.543974,0.270179,train
2019-01-01 03:00:00,20974.800758,18871.418601,4.693112e+06,380.561848,1466.329836,2103.388502,-281.391674,-191.565227,-24776.569759,-2.164379,...,1,3,3.0,327.608154,-323.038849,-1105.669434,-64278.093817,-39501.524058,1.594310,train
2019-01-01 04:00:00,20327.083333,18012.666667,5.032423e+06,711.911968,2391.657870,2314.666667,30.416667,74.416667,49254.136541,69.703951,...,1,4,4.0,319.256653,-290.810150,-634.733215,21847.402402,-27406.734139,0.556435,validation


In [34]:
PACE_Data.to_csv(os.path.join(best_model_dir, "PACE_Data_2019_2021_NN.with_coeff_preds.csv"))

In [35]:
len([val for val in PACE_Data.loc[:,"MEF"] if val <=0])

0

In [36]:
len([val for val in PACE_Data.loc[:,"MEF"] if val >600])

0

In [37]:
PACE_Data.loc[:,"MEF"].max()

np.float32(580.9651)

In [38]:
len([val for val in PACE_Data.loc[:,"MDF"] if val < -600])

40

In [39]:
PACE_Data.loc[:,"MDF"].min()

np.float32(-803.3059)